# 第4回：特徴量エンジニアリングの紹介

この回は3つのパートで構成します：**前処理をPipelineにまとめる ／ 特徴量を作る ／ 特徴量を選ぶ**。

**セルの動かし方**：各セル（灰色の枠）を選んで `Shift + Enter`（またはセル左の▷ボタン）を押すと実行できます。
**上から順に**実行してください。前のセルを飛ばすと、後のセルでエラーになります。

**AIと一緒に進める**：分からないコードは、セル全体ではなく気になる数行をM365 CopilotなどのAIへ貼り、
説明や修正を相談します。ただし、提案されたコードは必ず実行結果を見て確かめます。

まず「基本」と「演習」を進めます。「補足」は必要に応じて読み、
「発展（任意）」「追加演習（任意）」「自由課題（任意）」は飛ばしても構いません。


In [ ]:
# 【準備セル】教材フォルダの場所を自動で見つけます。中身は今は理解しなくてOK、そのまま実行してください。
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回で扱うこと

前処理をPipelineへ安全にまとめ、化学知識から特徴量を作り、作った特徴量を安全に選びます。

### 進め方

この回は3つのパートに分かれています。パート1から順に「基本」と「演習」を進めてください。
1日で終える必要はありません。「発展（任意）」と「追加演習（任意）」は、余裕がある場合だけ取り組みます。

### 用語について

初めて出る用語は、その用語を使うセルで説明します。ここでまとめて暗記する必要はありません。

> **実行前の30秒予想**：各パートの問いに、今の言葉で仮の答えを書いてから始めます。


---

# パート1：前処理をPipelineにまとめる

**このパートの問い：数値列とカテゴリ列を、安全に同じモデルへ入れるにはどうするか。**


## なぜ「Pipeline」が必要なのか

これまで欠損を`fillna`で埋めたり、数値だけを使ったりしてきました。実データでは**数値列と
カテゴリ列（文字）が混在**し、それぞれ別の下ごしらえが要ります。

- 数値列 → 欠損を埋める＋尺度を揃える（標準化）
- カテゴリ列 → 欠損を埋める＋数値へ変換（One-Hot：各カテゴリを0/1の列にする）

これらを手作業でやると、**第2回パート3で学んだ前処理リーク**（検証情報の漏れ）を起こしがちです。そこで
`Pipeline`と`ColumnTransformer`を使い、**前処理からモデルまでを1つの部品**にまとめます。こうすると
交差検証や予測のたびに、前処理が正しく分割の内側で学習されます。

まず数値列・カテゴリ列を決め、学習/検証に分けます。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

numeric = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
categorical = ["solvent", "catalyst", "scaffold_group"]
X = df[numeric + categorical]
y = df["active"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)


## 演習：列ごとの前処理を組み立てて、モデルまで繋ぐ

各部品の役割：

- `numeric_process`：数値列の下ごしらえ（欠損補完→標準化）を並べた小さなPipeline。
- `categorical_process`：カテゴリ列の下ごしらえ（欠損補完→One-Hot）。
- `ColumnTransformer`：「この列たちには数値処理、あの列たちにはカテゴリ処理」と**列ごとに担当を割り当てる**部品。
- 最後に`Pipeline([("前処理", preprocess), ("予測", ロジスティック回帰)])`で**前処理＋モデルを一体化**。

`model.fit`一発で、前処理もモデルもまとめて学習されます。


In [ ]:
numeric_process = Pipeline([
    ("欠損補完", SimpleImputer(strategy="median")),
    ("標準化", StandardScaler()),
])
categorical_process = Pipeline([
    ("欠損補完", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore")),
])
preprocess = ColumnTransformer([
    ("数値列", numeric_process, numeric),
    ("カテゴリ列", categorical_process, categorical),
])
model = Pipeline([("前処理", preprocess), ("予測", LogisticRegression(max_iter=1000))])
model.fit(X_train, y_train)
print(classification_report(y_valid, model.predict(X_valid), target_names=["非活性", "活性"]))


### 出力の読み方

`classification_report`は、クラスごとにprecision・recall・F1と件数(support)を並べた総合成績表です。

- **活性クラスの行**を重点的に見ます（少数派で難しいため）。第3回パート2で学んだとおり、accuracyより各クラスのrecall/precisionが実態を映します。
- 大事なのは点数そのものより、**文字列カテゴリを含む表をエラーなく1つのモデルへ通せた**こと。手作業のOne-Hotより安全で短いです。

補足：ここでは`scaffold_group`（化合物系列）もカテゴリ列の例として入れていますが、第2回パート3のとおり本来は
**系列を跨がない分割（GroupKFold）とセットで扱うべき列**です。この回はPipelineの組み方の説明が目的なので
乱数分割のまま使っていますが、実データで系列をカテゴリ特徴量にするときは、この点に注意してください。


## 未知カテゴリが来ても止まらない

本番では、学習時に無かった溶媒名が来ることがあります。`OneHotEncoder(handle_unknown="ignore")`の
おかげで、未知カテゴリでもエラーにならず予測できます。わざと存在しない溶媒名を入れて確かめます。


In [ ]:
unknown = X_valid.iloc[[0]].copy()
unknown["solvent"] = "New-Solvent"
print("未知カテゴリを含む予測:", model.predict(unknown)[0])


### 出力の読み方

エラーで止まらず予測が返れば成功です。`handle_unknown="ignore"`が無いと、未知カテゴリで例外が出て
本番が止まります。ここで身につけてほしいのは、**「本番で起きうる入力」を想定して前処理を設計する**という実務感覚です。


## 補足：変換後は列が増える。その姿を見る

One-Hotはカテゴリごとに0/1の列を作るので、**列数が増えます**。`get_feature_names_out`で変換後の列名を、
`transform`で実際の数値を確認し、Pipelineの中で何が起きているかを可視化します。


In [ ]:
names = model.named_steps["前処理"].get_feature_names_out()
transformed = model.named_steps["前処理"].transform(X_train.head(3))
if hasattr(transformed, "toarray"):
    transformed = transformed.toarray()
print("元の列数:", X_train.shape[1], "→ 変換後:", transformed.shape[1])
pd.DataFrame(transformed, columns=names, index=X_train.head(3).index).iloc[:, :10].round(2)


### 出力の読み方

- **元の列数 → 変換後**で列が増えているのは、カテゴリがOne-Hotで展開されたため。列名に`カテゴリ列__solvent_EtOH`のような名前が付きます。
- 数値列は標準化され、**平均0付近・小さめの値**になっています。One-Hot列は0か1。「モデルが実際に見ている数字」はこの姿です。

## 変更して確認

数値の欠損補完を`median`から`mean`へ変え、成績を比べます。変更は`SimpleImputer(strategy=...)`の**1か所だけ**。Pipelineだと変更点が1か所に集約され、実験が管理しやすくなります。


## 発展（任意）：自作の前処理を作り、前処理も探索対象にする

sklearnに用意された変換だけでなく、**自分の化学知識を前処理として書く**ことができます。また、
「どの補完戦略が良いか」のような前処理の選択も、モデルの設定と同じく**交差検証で選べます**。


### 自作変換器：`fit`と`transform`を持つ部品を書く

`BaseEstimator, TransformerMixin`を継承し、`fit`（学習することがあれば覚える）と`transform`（変換する）を
実装すれば、**Pipelineに差し込める自分だけの前処理**になります。ここでは「分子量あたりのTPSA」と
「最適温度からの距離」を足す変換器を作ります。


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np

class ChemRatioFeatures(BaseEstimator, TransformerMixin):
    "分子量あたりのTPSAと、最適温度78℃からの距離を足す自作変換器。"
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        X["tpsa_per_mw"] = X["tpsa"] / X["molecular_weight"].replace(0, np.nan)
        X["temp_distance"] = (X["temperature_c"] - 78).abs()
        return X

ChemRatioFeatures().fit_transform(df[["tpsa", "molecular_weight", "temperature_c"]].head()).round(3)


### 出力の読み方

元の3列に、新しい2列（`tpsa_per_mw`・`temp_distance`）が加わっています。`fit`は何も学習せず自身を返す
だけ（この変換は統計量を使わないため）。この形にしておくと、`Pipeline`へ入れて**分割の内側で**適用でき、
第4回パート2の特徴量設計をリークなく行えます。


### 前処理の設定を`GridSearchCV`で選ぶ

`Pipeline`の各部品の設定には`前処理__数値列__欠損補完__strategy`のように**アンダースコア2つ**で
辿り着けます。この記法を使い、補完戦略（median/mean）を交差検証で比較して自動選択します。


In [ ]:
from sklearn.model_selection import GridSearchCV

grid_pipe = Pipeline([("前処理", preprocess), ("予測", LogisticRegression(max_iter=1000))])
param_grid = {"前処理__数値列__欠損補完__strategy": ["median", "mean"]}
search = GridSearchCV(grid_pipe, param_grid, cv=5, scoring="f1")
search.fit(X_train, y_train)
print("最良設定:", search.best_params_)
print("最良CV F1:", round(search.best_score_, 3))


### 出力の読み方

`best_params_`が選ばれた補完戦略、`best_score_`がそのときの交差検証F1です。ポイントは、**前処理も
モデル設定と同じ土俵で、リークなく比較・選択できる**こと。Pipelineにまとめておいたからこそ可能になります。


## 追加演習（任意）

Pipelineをさらに実務的に使い込みます。90分の外の自習向けです。まず`make_column_selector`で、
**列の型（数値/文字）から自動で担当を振り分ける**書き方。列名を手で並べる手間が消えます。


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_selector, make_column_transformer

auto_pre = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), make_column_selector(dtype_include="number")),
    (make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), make_column_selector(dtype_include="object")),
)
auto_model = make_pipeline(auto_pre, LogisticRegression(max_iter=1000)).fit(X_train, y_train)
print("列の型で自動振り分けした検証精度:", round(auto_model.score(X_valid, y_valid), 3))


### 出力の読み方

`make_column_selector(dtype_include="number")`が数値列を、`"object"`が文字列列を自動で拾います。列が
増減しても書き換え不要。実データで列数が多いときに効きます。`.score`は分類では既定でaccuracyを返します。


### 前処理とモデルを「まとめて」探索する

前処理の設定とモデルのハイパーパラメータを、1つの`GridSearchCV`で同時に探します。すべてPipelineの
内側なので、リークなく公平に比較できます。


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

full = Pipeline([("前処理", preprocess), ("予測", RandomForestClassifier(random_state=42))])
grid = {
    "前処理__数値列__欠損補完__strategy": ["median", "mean"],
    "予測__max_depth": [4, 6, None],
    "予測__n_estimators": [200, 300],
}
search = GridSearchCV(full, grid, cv=5, scoring="f1")
search.fit(X_train, y_train)
print("最良設定:", search.best_params_)
print("最良CV F1:", round(search.best_score_, 3))


### 出力の読み方

前処理（補完戦略）とモデル（深さ・木の本数）の**最良の組み合わせ**が一度に選ばれます。組合せは
2×3×2=12通り×5分割=60回の学習。前処理も探索対象にできるのが、Pipeline最大の利点です。


### 学習済みPipelineを保存して再利用する

選ばれた最良のPipelineを`joblib`で保存し、読み直しても同じ予測になることを確かめます。前処理ごと
保存されるので、配布先は`predict`するだけです（第5回パート3の永続化の先取り）。


In [ ]:
import joblib
import numpy as np

path = ROOT / "workspace" / "pipeline_09.joblib"
joblib.dump(search.best_estimator_, path)
loaded = joblib.load(path)
assert np.array_equal(search.best_estimator_.predict(X_valid), loaded.predict(X_valid)), "保存前後で予測が不一致"
print("保存・読込で同じ予測:", path)


### 出力の読み方

`assert`が通れば、前処理込みのPipelineが丸ごと保存・復元できたということ。「モデルだけ保存して前処理を
忘れる」という実務で頻発する事故を、Pipeline化で防げます。


---

# パート2：特徴量を作る

**このパートの問い：研究者の知識を、モデルへ渡せる形にするにはどうするか。**


## 特徴量設計＝あなたの化学知識をモデルへ渡す

モデルは与えられた列しか見ません。**「最適温度から離れるほど収率が落ちる」**という知識を持っていても、
`temperature_c`の生の値だけでは、モデルがその山型を学ぶのは大変です。そこで、知識を**計算式**にして
新しい列（特徴量）として渡します。これが特徴量設計です。

鉄則が2つあります。
1. **予測時点で計算できること**（第2回パート2。実験後の値から作らない）。
2. **追加の効果は、同じ検証条件で前後比較して確かめる**（思い込みで良し悪しを決めない）。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


## 演習：仮説を計算式にする

2つの仮説を式にします。**「最適温度78℃からの距離」**（離れるほど収率減、という山型を直接表す）と、
**「単位時間あたりの濃度」**（濃度と時間の兼ね合い）。どちらも計画時に計算できる値です。


In [ ]:
engineered = df.copy()
engineered["temperature_distance"] = (engineered["temperature_c"] - 78).abs()
engineered["concentration_per_hour"] = engineered["concentration_m"] / engineered["reaction_time_h"]
engineered[["temperature_c", "temperature_distance", "concentration_per_hour"]].head()


### 読みどころ

`temperature_distance`は、78℃から上下どちらに離れても大きくなる値（絶対値）。第2回パート1で見た「温度と収率の
山型」を、モデルにとって学びやすい**単調な形**に翻訳しています。生の温度より効くかどうかは、次で検証します。


## アブレーション：追加の効果を「同じ条件」で確かめる

**アブレーション**とは、要素を足し引きして寄与を測る比較のこと。特徴量を追加する前後で、
**同じモデル・同じ交差検証**でMAEを比べます。これをやらずに「良さそうだから採用」は禁物です。


In [ ]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

base = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
added = [*base, "temperature_distance", "concentration_per_hour"]
cv = KFold(5, shuffle=True, random_state=42)
for label, cols in {"追加前": base, "追加後": added}.items():
    est = make_pipeline(SimpleImputer(strategy="median"), RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42))
    scores = cross_val_score(est, engineered[cols], engineered["yield_pct"], cv=cv, scoring="neg_mean_absolute_error")
    print(f"{label}: MAE={-scores.mean():.3f} ± {scores.std():.3f}")


### 出力の読み方

- 「追加後」のMAEが「追加前」より**下がっていれば**、その特徴量は効いています。**ばらつき(±)より大きく**下がっているかも見ます（±の中の差は誤差かも）。
- 効かない・悪化することもあります。それも立派な結果。**「この仮説はこのモデルには効かなかった」**と分かるのが検証の価値です。悪化した実験も記録します（第4回パート3）。


## 補足：関連の強い特徴量を選ぶ（相互情報量）

特徴量が増えると、効かない列がノイズになることも。**相互情報量（第2回パート1）**で目的変数との関連が強い順に
並べ、上位k個を選びます。相関と違い、山型のような非線形の関連も拾えます。


In [ ]:
from functools import partial
from sklearn.feature_selection import SelectKBest, mutual_info_regression

# random_stateを固定しないとMIの推定値は実行ごとに変わる（第2回パート1と同じ作法）
mi_score = partial(mutual_info_regression, random_state=42)
sel_data = engineered[added].fillna(engineered[added].median())
selector = SelectKBest(mi_score, k=4).fit(sel_data, engineered["yield_pct"])
pd.DataFrame({"特徴量": added, "MIスコア": selector.scores_, "選択": selector.get_support()}).sort_values("MIスコア", ascending=False).round(3)


### 出力の読み方（結果は素直に受け止める）

MIスコアの高い順に並び、上位4つに「選択=True」が付きます。ここで大事なのは、**自作の`temperature_distance`が
必ず上位に来るとは限らない**ことです。実際、このデータの単変量MIでは上位に来ないことがあります。相互情報量は
**1列ずつ単独で**目的変数との関連を測るため、「他の列と組み合わせて効く」種類の特徴量を低く見積もることがあります。
思い込みで良し悪しを決めず、数字を見る。そして次の発展（任意）/追加演習で、**別の見方（並べ替え重要度）だと結論が
変わる**ことを実際に確かめます。`random_state`を固定しているのは、固定しないとMIの推定値が毎回変わるためです。


## 自由課題（任意）：RDKitでSMILESから記述子を計算する

分子量やLogPは、本来は分子構造（SMILES）から計算できます。RDKitが入っていれば、エタノールの
記述子を実際に計算してみます。無い環境では自動でスキップし、計算済みの列で本編を進められます。


In [ ]:
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, Crippen
    molecule = Chem.MolFromSmiles("CCO")
    print("エタノールの分子量:", round(Descriptors.MolWt(molecule), 2))
    print("エタノールのLogP:", round(Crippen.MolLogP(molecule), 2))
except ImportError:
    print("RDKitは任意（uv sync --extra chemistry）。計算済みmolecular_weight/logp/tpsaで本編を進められます。")


### 読みどころ

RDKitが動けば、SMILES（`CCO`＝エタノール）から分子量やLogPが再現されます。**「記述子＝構造から計算できる
特徴量」**だと腹落ちします。RDKitは発展扱いなので、無くても計算済みの列で全く問題ありません。


---

# パート3：特徴量を選ぶ

**このパートの問い：作った特徴量は、本当に信頼して使ってよいか。**


## 作った特徴量を、安全に選ぶ

前のパートで特徴量を作りました。ここでは2つの問いを扱います。**「強力だがリークしやすい作り方を、
安全に使えるか」**、そして**「たくさん作った特徴量から、どれを残すか」**です。

強力だが**リークしやすい**特徴量の代表が**target encoding**（カテゴリを目的変数の平均で置き換える）です。
やり方を誤ると、第2回パート3で学んだリークを自ら仕込むことになります。安全なやり方を身につけます。


### target encoding：全データ平均は「リーク」、OOFなら安全

「系列ごとの平均収率」を特徴量にしたいとします。**全データの平均**で作ると、各行の答えが自分の特徴量に
混ざりリークします。正しくは、第2回パート3の交差検証と同じ発想で、**その行を含まない分割の平均**で作ります
（OOF＝out-of-fold）。両者でMAEを比べ、リークが楽観を生むことを確かめます。


In [ ]:
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import Ridge

def oof_target_encode(frame, col, target, n_splits=5, seed=42):
    "分割の内側で平均を学習するリーク安全なtarget encoding。"
    encoded = pd.Series(index=frame.index, dtype=float)
    global_mean = frame[target].mean()
    for tr, va in KFold(n_splits, shuffle=True, random_state=seed).split(frame):
        means = frame.iloc[tr].groupby(col)[target].mean()
        encoded.iloc[va] = frame.iloc[va][col].map(means).fillna(global_mean).to_numpy()
    return encoded

leaky = df["scaffold_group"].map(df.groupby("scaffold_group")["yield_pct"].mean())
safe = oof_target_encode(df, "scaffold_group", "yield_pct")
num_cols = ["temperature_c", "concentration_m", "logp"]
X_num = df[num_cols].fillna(df[num_cols].median())
for label, enc in {"リークあり(全データ平均)": leaky, "OOF(安全)": safe}.items():
    feats = X_num.assign(scaffold_te=enc.to_numpy())
    scores = cross_val_score(Ridge(), feats, df["yield_pct"], cv=5, scoring="neg_mean_absolute_error")
    print(f"{label}: MAE={-scores.mean():.3f}")
print("リークありは楽観的に見えることがある。実運用の性能はOOFに近い。")


### 出力の読み方

「リークあり」のMAEが「OOF」より**小さく（良く）見える**ことがあります。しかしそれは幻。本番では
その行の答えは手に入りません。**実運用の実力はOFFの側**。強力な特徴量ほど、作り方のリークに注意します。


### RFECV：交差検証つきで特徴量を絞り込む

`RFECV`は、重要度の低い特徴量を1つずつ削りながら交差検証し、**性能が最も良くなる特徴量の組**を
自動で選びます。人手の取捨選択より客観的です。ここでは**係数の大きさで重要度を測る線形モデル(Ridge)**で
回します（後述のとおり、木モデルはノイズに強すぎてRFECVが列を削らないことが多いため）。尺度をそろえてから
かけます。


In [ ]:
import numpy as np
from sklearn.feature_selection import RFECV
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

# わざと「無意味な列」を混ぜて、RFECVがそれを削れるかを確かめる
rng = np.random.default_rng(0)
rfe_data = engineered[added].fillna(engineered[added].median()).copy()
rfe_data["noise"] = rng.normal(size=len(rfe_data))     # 目的変数と無関係な乱数列
rfe_data["logp_copy"] = rfe_data["logp"]                # 既存列の複製（冗長）
scaled = pd.DataFrame(StandardScaler().fit_transform(rfe_data), columns=rfe_data.columns, index=rfe_data.index)
rfecv = RFECV(Ridge(alpha=1.0), cv=5, scoring="neg_mean_absolute_error", min_features_to_select=2)
rfecv.fit(scaled, engineered["yield_pct"])
print("元の列数:", scaled.shape[1], "→ 選ばれた列数:", rfecv.n_features_)
pd.DataFrame({"特徴量": rfe_data.columns, "残す": rfecv.support_, "順位": rfecv.ranking_}).sort_values("順位")


### 出力の読み方

- `残す=True`が採用列、`順位=1`が最重要グループ。**わざと混ぜた`noise`（乱数）と`logp_copy`（複製）が削られていれば**、RFECVが「役に立たない列を見抜いて外す」働きをしていると確認できます。
- 木モデル(RandomForest)ではなく線形モデル(Ridge)を使ったのは、**木モデルはノイズ列があっても性能が落ちにくく、RFECVが何も削らないことが多い**ため。**推定器を変えると選択結果も変わる**。特徴量選択も「どの手法で測るか」に依存する、という点も併せて押さえます。
- 選択も交差検証の内側で行うことで、選びすぎ（過学習）を避けています。


## 発展（任意）：特徴量の作り方をもう2つ

ここからは経験者・自習向けの発展です。**交互作用特徴量**（2つの列の掛け算で「組み合わせの効果」を
表す）と、連続値を区間に区切る**ビニング**を扱います。


In [ ]:
from sklearn.preprocessing import PolynomialFeatures

cols = ["temperature_c", "concentration_m"]
pair = engineered[cols].fillna(engineered[cols].median())
inter = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
out = inter.fit_transform(pair)
display(pd.DataFrame(out, columns=inter.get_feature_names_out(), index=pair.index).head())


### 出力の読み方

元の2列に加え、`temperature_c concentration_m`（掛け算）の列ができます。`interaction_only=True`なので
二乗は作らず組み合わせだけ。「片方が高いときだけもう片方が効く」ような関係を、モデルへ渡せます。


### 連続値を区間に区切る（ビニング）

温度のような連続値を4区間に区切ると、非線形な効果を扱いやすくなったり、解釈しやすくなったりします。
`KBinsDiscretizer`（分位点で等件数に区切る）を使い、区間ごとの平均収率を見ます。


In [ ]:
from sklearn.preprocessing import KBinsDiscretizer

temp = engineered[["temperature_c"]].fillna(engineered["temperature_c"].median())
binner = KBinsDiscretizer(n_bins=4, encode="ordinal", strategy="quantile")
engineered["temp_bin"] = binner.fit_transform(temp).astype(int)
display(engineered.groupby("temp_bin")["yield_pct"].mean().round(1))


### 出力の読み方

区間0（低温）〜3（高温）ごとの平均収率が出ます。中間の区間で収率が高い（山型）なら、第2回パート1で見た
温度の効果と一致。ビニングは効果を見せやすい一方、情報を捨てる面もあるので、元の連続値と併用も検討します。


## 追加演習（任意）

作った特徴量の効き目を、並べ替え重要度で確かめます。第1・第4回パート1で使った並べ替え重要度を、
この回で作った特徴量を含めた全体に適用します。自作特徴量が上位に来るかを、holdoutで公平に確認します。


In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

Xe = engineered[added].fillna(engineered[added].median())
Xtr, Xva, ytr, yva = train_test_split(Xe, engineered["yield_pct"], test_size=0.25, random_state=42)
rf = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42).fit(Xtr, ytr)
perm = permutation_importance(rf, Xva, yva, scoring="neg_mean_absolute_error", n_repeats=15, random_state=42)
pd.DataFrame({"特徴量": added, "重要度": perm.importances_mean}).sort_values("重要度", ascending=False).round(3)


### 出力の読み方：3つの見方が食い違うのは正常

ここでは`temperature_distance`が**上位に来ることがあります**。ところが同じ回の基本では、相互情報量(MI)で
同じ列が**下位**、アブレーションでは追加しても**MAEがほとんど改善しない**。3つの見方で結論が食い違います。
矛盾ではなく、**それぞれ別の問いに答えているから**です。

- **アブレーション**：その列を入れるか抜くかで最終性能がどう動くか。他の列で代用が効くと、抜いても悪化せず「効果なし」に見える。
- **相互情報量**：その列を単独で見たときの関連の強さ。組み合わせて効く効果は測れない。
- **並べ替え重要度**：学習済みモデルが実際にその列に依存しているか。`temperature_distance`は`temperature_c`から作った相関の強い列なので、モデルがどちらを使うかで重要度が振れやすい。

教訓は2つ。**(1) 1つの指標だけで特徴量の良し悪しを断じない。(2) 元の列と強く相関する派生列（今回の距離特徴量）は、
重要度が不安定になりやすい。** 「作る→交差検証で効果を確かめる→複数の見方で解釈する」という一巡こそが、
思い込みを避ける特徴量設計です。


---

## よくある誤り

- 全データ平均で欠損補完する
- カテゴリを意味のない大小関係へ変換する
- 本番の未知カテゴリでエラーになる
- 意味を説明できない特徴量を大量追加する
- 追加前後で分割やモデルも変える
- RDKitが無いと動かない前提でコードを書く
- 目的変数由来の値を全データで作って特徴量にする
- 特徴量選択を分割の外側で行う
- 重要度の高さだけで採用可否を決め、意味を確認しない

## 自習（任意・30〜60分）

- 分子量あたりのTPSAを作る自作変換器を書き、Pipelineへ組み込む
- 数値標準化の有無と補完戦略をGridSearchCVで比較する
- 比・差以外の組み合わせ特徴量を1つ作り、MAEの変化を記録する
- RDKit記述子を使う場合と計算済み記述子表を使う場合で結果を比べる
- 自作KFold target encodingの有無でMAEを比較する
- RFECVで残った特徴量と、化学的な解釈を突き合わせる

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. 自作変換器に最低限必要なメソッドは何か
2. 前処理をPipelineへ入れるとリークがなぜ防げるか
3. get_feature_names_outは何に使うか
4. その特徴量はいつ計算できるか
5. アブレーションとは何を確かめる操作か
6. RDKit記述子はどんな情報から計算されるか
7. target encodingでリークを防ぐ手順は何か
8. 特徴量選択も交差検証の内側で行う理由は何か
9. 並べ替え重要度が0付近の特徴量をどう扱うか

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
